# RIASEC Dataset Cleaning and Preprocessing

This notebook transforms the raw **OpenPsychometrics RIASEC dataset** into a quality-controlled dataset for vocational-domain classification.

The pipeline covers:

1. Loading and validating the raw dataset
2. Cleaning the 48 RIASEC responses
3. Applying completeness and response-quality rules
4. Calculating the six RIASEC dimension scores
5. Creating Holland Codes
6. Cleaning and standardizing free-text majors
7. Mapping majors to broad academic categories
8. Exporting the final datasets used for machine-learning experiments

**Dataset source:** https://openpsychometrics.org/_rawdata/

The mapping stage is intentionally conservative: ambiguous and rare free-text majors are left unmapped rather than being forced into a potentially incorrect category. This reduces label noise in the supervised target.


In [1]:
# ============================================================
# RIASEC DATASET — COMPLETE CLEANING & PREPROCESSING PIPELINE
# ============================================================

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# 0. LOAD ORIGINAL RAW DATA
# ------------------------------------------------------------

RAW_FILE = "/content/data.csv"   # <-- CHANGE ONLY IF YOUR FILE HAS ANOTHER NAME

# The OpenPsychometrics file is tab-separated
df = pd.read_csv(
    RAW_FILE,
    sep="\t",
    low_memory=False
)

print("=" * 70)
print("0. RAW DATA")
print("=" * 70)
print("Raw shape:", df.shape)

# Optional useless export column
if "Unnamed: 93" in df.columns:
    df = df.drop(columns=["Unnamed: 93"])

print("Shape after removing Unnamed column if present:", df.shape)


# ------------------------------------------------------------
# 1. DEFINE THE 48 RIASEC QUESTIONS
# ------------------------------------------------------------

riasec_cols = (
    [f"R{i}" for i in range(1, 9)] +
    [f"I{i}" for i in range(1, 9)] +
    [f"A{i}" for i in range(1, 9)] +
    [f"S{i}" for i in range(1, 9)] +
    [f"E{i}" for i in range(1, 9)] +
    [f"C{i}" for i in range(1, 9)]
)

print("\nNumber of RIASEC questions:", len(riasec_cols))

# Make sure required columns exist
required_cols = riasec_cols + ["major", "testelapse"]

missing_required = [
    col for col in required_cols
    if col not in df.columns
]

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}"
    )


# ------------------------------------------------------------
# 2. CREATE RIASEC-ONLY WORKING DATAFRAME
# ------------------------------------------------------------

# IMPORTANT:
# df stays untouched from this point.
# All cleaning happens on riasec_df.

riasec_df = df[
    riasec_cols + ["major", "testelapse"]
].copy()

print("\n" + "=" * 70)
print("1. RIASEC WORKING DATASET")
print("=" * 70)

print("Shape:", riasec_df.shape)


# ------------------------------------------------------------
# 3. CONVERT RIASEC QUESTIONS TO NUMERIC
# ------------------------------------------------------------

riasec_df[riasec_cols] = (
    riasec_df[riasec_cols]
    .apply(pd.to_numeric, errors="coerce")
)


# ------------------------------------------------------------
# 4. HANDLE 0 VALUES
# ------------------------------------------------------------
# According to the codebook:
#
# 1 = Dislike
# 3 = Neutral
# 5 = Enjoy
#
# Valid scale = 1–5
# Therefore 0 is NOT a valid response.
# Treat 0 as missing.

zero_answers_before = (
    riasec_df[riasec_cols] == 0
).sum().sum()

people_with_zero = (
    riasec_df[riasec_cols] == 0
).any(axis=1).sum()

print("\n" + "=" * 70)
print("2. INVALID ZERO RESPONSES")
print("=" * 70)

print("Zero answers found:", zero_answers_before)
print("Respondents with at least one zero:", people_with_zero)

riasec_df[riasec_cols] = (
    riasec_df[riasec_cols]
    .replace(0, np.nan)
)


# ------------------------------------------------------------
# 5. COUNT MISSING ANSWERS
# ------------------------------------------------------------

riasec_df["missing_answers"] = (
    riasec_df[riasec_cols]
    .isna()
    .sum(axis=1)
)

print("\nMissing responses after converting 0 → NaN:")
print(
    riasec_df["missing_answers"]
    .value_counts()
    .sort_index()
    .head(20)
)


# ------------------------------------------------------------
# 6. COUNT ANSWERED QUESTIONS PER RIASEC DIMENSION
# ------------------------------------------------------------

dimensions = ["R", "I", "A", "S", "E", "C"]

answered_cols = []

for letter in dimensions:

    cols = [
        f"{letter}{i}"
        for i in range(1, 9)
    ]

    answered_col = f"{letter}_answered"

    riasec_df[answered_col] = (
        riasec_df[cols]
        .notna()
        .sum(axis=1)
    )

    answered_cols.append(answered_col)


# ------------------------------------------------------------
# 7. RIASEC COMPLETENESS RULE
# ------------------------------------------------------------
# Keep a respondent only if they answered at least
# 6 out of 8 questions in EVERY RIASEC dimension.

valid_riasec = (
    riasec_df[answered_cols] >= 6
).all(axis=1)

print("\n" + "=" * 70)
print("3. RIASEC COMPLETENESS")
print("=" * 70)

print("Valid respondents:", valid_riasec.sum())
print("Invalid respondents:", (~valid_riasec).sum())

# Remove only respondents failing this rule
riasec_df = (
    riasec_df[valid_riasec]
    .copy()
)

print("Rows remaining:", len(riasec_df))


# ------------------------------------------------------------
# 8. CALCULATE THE SIX RIASEC SCORES
# ------------------------------------------------------------
# Pandas mean ignores NaN.
# Because every person has >= 6/8 answers per dimension,
# each score remains reasonably supported.

for letter in dimensions:

    cols = [
        f"{letter}{i}"
        for i in range(1, 9)
    ]

    riasec_df[f"{letter}_score"] = (
        riasec_df[cols]
        .mean(axis=1)
    )

score_cols = [
    "R_score",
    "I_score",
    "A_score",
    "S_score",
    "E_score",
    "C_score"
]

print("\n" + "=" * 70)
print("4. RIASEC SCORES")
print("=" * 70)

print(
    riasec_df[score_cols]
    .describe()
    .round(3)
)

print(
    "\nMissing calculated scores:",
    riasec_df[score_cols]
    .isna()
    .sum()
    .sum()
)

print(
    "Scores below 1:",
    (riasec_df[score_cols] < 1)
    .sum()
    .sum()
)

print(
    "Scores above 5:",
    (riasec_df[score_cols] > 5)
    .sum()
    .sum()
)


# ------------------------------------------------------------
# 9. COMPLETION-TIME QUALITY FLAG
# ------------------------------------------------------------
# Flag — but DO NOT automatically remove:
#
# < 60 seconds = suspiciously fast
# > 3600 seconds = over 1 hour

riasec_df["suspicious_time"] = (
    (riasec_df["testelapse"] < 60) |
    (riasec_df["testelapse"] > 3600)
)

print("\n" + "=" * 70)
print("5. COMPLETION TIME")
print("=" * 70)

print(
    riasec_df["testelapse"]
    .describe()
    .round(2)
)

print(
    "\nSuspicious-time respondents:",
    riasec_df["suspicious_time"].sum()
)

print(
    "Under 60 seconds:",
    (riasec_df["testelapse"] < 60).sum()
)

print(
    "Over 1 hour:",
    (riasec_df["testelapse"] > 3600).sum()
)


# ------------------------------------------------------------
# 10. STRAIGHT-LINING CHECK
# ------------------------------------------------------------
# Straight-lining = exactly the same response across all
# answered RIASEC questions.

riasec_df["riasec_std"] = (
    riasec_df[riasec_cols]
    .std(axis=1)
)

riasec_df["straight_lining"] = (
    riasec_df["riasec_std"] == 0
)

print("\n" + "=" * 70)
print("6. STRAIGHT-LINING")
print("=" * 70)

print(
    "Straight-lining respondents:",
    riasec_df["straight_lining"].sum()
)

print("\nStraight-lining × suspicious time:")

print(
    pd.crosstab(
        riasec_df["straight_lining"],
        riasec_df["suspicious_time"]
    )
)


# ------------------------------------------------------------
# 11. LOW-QUALITY RESPONSE RULE
# ------------------------------------------------------------
# We decided NOT to remove:
# - suspicious time alone
# - straight-lining alone
#
# Remove only respondents showing BOTH signals.

riasec_df["low_quality"] = (
    riasec_df["straight_lining"] &
    riasec_df["suspicious_time"]
)

low_quality_count = (
    riasec_df["low_quality"].sum()
)

print("\nLow-quality respondents to remove:",
      low_quality_count)

riasec_df = (
    riasec_df[
        ~riasec_df["low_quality"]
    ]
    .copy()
)

print(
    "RIASEC-clean dataset after quality filtering:",
    len(riasec_df)
)


# ------------------------------------------------------------
# 12. BASIC MAJOR TEXT CLEANING
# ------------------------------------------------------------
# IMPORTANT:
# Do NOT standardize major names yet.
#
# This only:
# - lowercases
# - strips leading/trailing spaces
# - collapses repeated spaces

riasec_df["major_clean"] = (
    riasec_df["major"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

print("\n" + "=" * 70)
print("7. MAJOR AVAILABILITY")
print("=" * 70)

print(
    "Missing majors:",
    riasec_df["major_clean"]
    .isna()
    .sum()
)

print(
    "Empty majors:",
    (
        riasec_df["major_clean"]
        .fillna("")
        .str.strip()
        == ""
    ).sum()
)


# ------------------------------------------------------------
# 13. CREATE SUPERVISED MODEL DATASET
# ------------------------------------------------------------
# riasec_df:
#   everyone with usable RIASEC data
#
# model_df:
#   usable RIASEC + known major
#
# We keep both separately.

model_df = riasec_df[
    riasec_df["major_clean"].notna() &
    (
        riasec_df["major_clean"]
        .str.strip()
        != ""
    )
].copy()

print("\nRIASEC-clean dataset:", len(riasec_df))
print("RIASEC + major dataset:", len(model_df))


# ------------------------------------------------------------
# 14. CREATE HOLLAND CODE
# ------------------------------------------------------------
# Keep the continuous six scores for ML.
# Holland code is mainly useful for interpretation/display.

score_map = {
    "R": "R_score",
    "I": "I_score",
    "A": "A_score",
    "S": "S_score",
    "E": "E_score",
    "C": "C_score"
}

def get_holland_code(row):

    ranked = sorted(
        score_map.keys(),
        key=lambda x: row[score_map[x]],
        reverse=True
    )

    return "".join(ranked[:3])

model_df["holland_code"] = (
    model_df.apply(
        get_holland_code,
        axis=1
    )
)


# ------------------------------------------------------------
# 15. FINAL DATA QUALITY CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("8. FINAL QUALITY CHECK")
print("=" * 70)

print("Rows:", len(model_df))
print("Columns:", model_df.shape[1])

print(
    "\nValid raw RIASEC values:",
    sorted(
        [
            x
            for x in pd.unique(
                model_df[
                    riasec_cols
                ].values.ravel()
            )
            if pd.notna(x)
        ]
    )
)

print(
    "\nMissing raw RIASEC answers:",
    model_df[riasec_cols]
    .isna()
    .sum()
    .sum()
)

print(
    "Respondents with ≥1 missing raw answer:",
    model_df[riasec_cols]
    .isna()
    .any(axis=1)
    .sum()
)

print(
    "Minimum total answered questions:",
    model_df[riasec_cols]
    .notna()
    .sum(axis=1)
    .min()
)

print("\nAnswered questions per dimension:")

for letter in dimensions:

    column = f"{letter}_answered"

    print(
        f"{letter}: "
        f"min={model_df[column].min()}, "
        f"max={model_df[column].max()}"
    )

print(
    "\nMissing calculated RIASEC scores:",
    model_df[score_cols]
    .isna()
    .sum()
    .sum()
)

print(
    "Scores below 1:",
    (model_df[score_cols] < 1)
    .sum()
    .sum()
)

print(
    "Scores above 5:",
    (model_df[score_cols] > 5)
    .sum()
    .sum()
)

print(
    "\nSuspicious-time respondents:",
    model_df["suspicious_time"]
    .sum()
)

print(
    "Straight-lining respondents:",
    model_df["straight_lining"]
    .sum()
)

print(
    "Low-quality flag remaining:",
    model_df["low_quality"]
    .sum()
)

print(
    "\nMissing cleaned majors:",
    model_df["major_clean"]
    .isna()
    .sum()
)

print(
    "Empty cleaned majors:",
    (
        model_df["major_clean"]
        .str.strip()
        == ""
    ).sum()
)

print(
    "Unique cleaned majors:",
    model_df["major_clean"]
    .nunique()
)

print(
    "\nCompletely identical rows:",
    model_df
    .duplicated()
    .sum()
)

print(
    "Duplicate 48-answer + major combinations:",
    model_df
    .duplicated(
        subset=riasec_cols + ["major_clean"]
    )
    .sum()
)


# ------------------------------------------------------------
# 16. MAJOR DISTRIBUTION
# ------------------------------------------------------------

major_counts = (
    model_df["major_clean"]
    .value_counts()
)

print("\n" + "=" * 70)
print("9. MAJOR DISTRIBUTION")
print("=" * 70)

print("\nTop 30 majors:")

print(
    major_counts
    .head(30)
)

print(
    "\nMajors appearing exactly once:",
    (major_counts == 1).sum()
)

print(
    "Majors appearing < 5 times:",
    (major_counts < 5).sum()
)

print(
    "Majors appearing < 10 times:",
    (major_counts < 10).sum()
)

print(
    "Majors appearing >= 10 times:",
    (major_counts >= 10).sum()
)

print(
    "Majors appearing >= 50 times:",
    (major_counts >= 50).sum()
)

print(
    "Majors appearing >= 100 times:",
    (major_counts >= 100).sum()
)


# ------------------------------------------------------------
# 17. EXPECTED CHECKPOINT VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("10. CHECKPOINT VERIFICATION")
print("=" * 70)

EXPECTED_RAW = 145828
EXPECTED_MODEL = 92742

print("Expected original rows:", EXPECTED_RAW)
print("Actual original rows:", len(df))

print()

print("Expected final model rows:", EXPECTED_MODEL)
print("Actual final model rows:", len(model_df))

if len(df) == EXPECTED_RAW:
    print("✅ Original dataset size is correct.")
else:
    print("⚠️ WARNING: Original dataset size differs from expected.")

if len(model_df) == EXPECTED_MODEL:
    print("✅ Final model dataset size is correct.")
else:
    print(
        "⚠️ WARNING: Final model dataset size does NOT equal 92,742."
    )


# ------------------------------------------------------------
# 18. SAVE PROTECTED CHECKPOINTS
# ------------------------------------------------------------
# Only save the official checkpoint if the expected final row
# count is correct. This prevents Run All from overwriting our
# good checkpoint with an accidentally filtered dataframe.

if len(model_df) == EXPECTED_MODEL:

    MODEL_CHECKPOINT = (
        "/content/riasec_model_checkpoint_92742.csv"
    )

    RIASEC_CHECKPOINT = (
        "/content/riasec_clean_all_respondents.csv"
    )

    model_df.to_csv(
        MODEL_CHECKPOINT,
        index=False
    )

    riasec_df.to_csv(
        RIASEC_CHECKPOINT,
        index=False
    )

    print("\n✅ CHECKPOINTS SAVED")

    print("Model checkpoint:")
    print(MODEL_CHECKPOINT)

    print("\nRIASEC-only checkpoint:")
    print(RIASEC_CHECKPOINT)

else:

    print(
        "\n❌ CHECKPOINT NOT SAVED because "
        "the final row count is unexpected."
    )


# ------------------------------------------------------------
# 19. VERIFY SAVED CHECKPOINT
# ------------------------------------------------------------

if len(model_df) == EXPECTED_MODEL:

    verification_df = pd.read_csv(
        MODEL_CHECKPOINT,
        low_memory=False
    )

    print("\n" + "=" * 70)
    print("11. SAVED FILE VERIFICATION")
    print("=" * 70)

    print(
        "Reloaded checkpoint shape:",
        verification_df.shape
    )

    if len(verification_df) == EXPECTED_MODEL:
        print(
            "✅ Saved checkpoint successfully verified."
        )
    else:
        raise ValueError(
            "Saved checkpoint row count is incorrect!"
        )


# ------------------------------------------------------------
# 20. EXPORT MAJOR LIST WITH COUNTS
# ------------------------------------------------------------

major_list = (
    model_df["major_clean"]
    .value_counts()
    .reset_index()
)

major_list.columns = [
    "major",
    "number_of_samples"
]

MAJOR_LIST_FILE = (
    "/content/all_majors_with_counts.csv"
)

major_list.to_csv(
    MAJOR_LIST_FILE,
    index=False
)

print("\nMajor list saved:")
print(MAJOR_LIST_FILE)


# ------------------------------------------------------------
# DONE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)

print(
    "From now on use model_df for major standardization."
)

print(
    "Do NOT overwrite model_df during experiments."
)

print(
    "Always create a copy:"
)

print(
    "major_df = model_df.copy()"
)

0. RAW DATA
Raw shape: (145828, 94)
Shape after removing Unnamed column if present: (145828, 93)

Number of RIASEC questions: 48

1. RIASEC WORKING DATASET
Shape: (145828, 50)

2. INVALID ZERO RESPONSES
Zero answers found: 21435
Respondents with at least one zero: 10064

Missing responses after converting 0 → NaN:
missing_answers
0     135764
1       7390
2       1652
3        489
4        172
5         72
6         36
7         26
8          7
9          7
10        10
11         5
12         3
13         4
14         2
15         1
16         3
17         4
18         1
19         1
Name: count, dtype: int64

3. RIASEC COMPLETENESS
Valid respondents: 145562
Invalid respondents: 266
Rows remaining: 145562

4. RIASEC SCORES
          R_score     I_score     A_score     S_score     E_score     C_score
count  145562.000  145562.000  145562.000  145562.000  145562.000  145562.000
mean        2.090       3.006       2.871       3.267       2.570       2.394
std         0.891       1.046   

# ============================================================
# MAJOR CLEANING & STANDARDIZATION
# ============================================================

## Major cleaning strategy

The original `major` field is free text, so it contains spelling variations, abbreviations, multilingual responses, school-only answers, undecided responses, and other noise.

This section:

- marks clearly invalid responses as `invalid`;
- keeps recoverable responses when a high-confidence normalization is possible;
- standardizes only high-confidence aliases;
- preserves multilingual majors when they contain valid Unicode letters;
- avoids forcing ambiguous labels into a category.

All major-cleaning experiments use `major_df = model_df.copy()` so the protected preprocessing checkpoint is not overwritten.


In [2]:
major_df = model_df.copy()

print("Starting rows:", len(major_df))
print("Unique majors:", major_df["major_clean"].nunique())

Starting rows: 92742
Unique majors: 11759


In [3]:
major_df["major_status"] = "valid"

major_df["major_standardized"] = (
    major_df["major_clean"].copy()
)

In [4]:
invalid_exact = {
    "no",
    "none",
    "undecided",
    "undeclared",
    "na",
    "n/a",
    "n/a?",
    "n.a.",
    "n.a",
    "n,a",
    "n>a",
    "n/n",
    "unknown",
    "not applicable",
    "not sure",
    "not sure yet",
    "not decided",

    "i dont know",
    "i dont know yet",
    "i dont know.",
    "dont know",
    "dont know yet",
    "im not sure",
    "im not sure yet",
    "i am not sure",

    "idk",
    "i dk",
    "idek",
    "duno",

    "nil",
    "nil-",
    "nope",
    "non",

    "-", "--", "---", "–",
    "?", "??", "????",
    "...", "/", ";",

    "0", "1", "3", "4", "5",
    "10", "22", "42", "46",
    "50", "55", "932",
    "1212", "2323",

    "sdfg",
    "ggg",
    "blah",
    "huh?"
}

major_df.loc[
    major_df["major_clean"].isin(invalid_exact),
    "major_status"
] = "invalid"

In [5]:
no_university_patterns = [
    "not attended",
    "never attended",
    "have not attended",
    "not yet attended",
    "not in college",
    "still in high school",
    "less than high school"
]

pattern = "|".join(no_university_patterns)

mask = major_df["major_clean"].str.contains(
    pattern,
    case=False,
    na=False,
    regex=True
)

major_df.loc[
    mask,
    "major_status"
] = "invalid"

In [6]:
recoverable_map = {
    "business undecided": "business",
    "undecided business": "business",
    "undecided/business": "business",
    "dont know maybe buissness": "business",

    "undecided science": "science",
    "undeclared nursing": "nursing",
    "music (undeclared)": "music",

    "undecided maybe psychology": "psychology",
    "undecided maybe anthrozoology": "anthrozoology",

    "arts (not sure which specific one yet)": "arts",
    "liberal arts, no major": "liberal arts"
}

for original, replacement in recoverable_map.items():

    mask = major_df["major_clean"] == original

    major_df.loc[
        mask,
        "major_status"
    ] = "recoverable"

    major_df.loc[
        mask,
        "major_standardized"
    ] = replacement

In [7]:
print(major_df["major_status"].value_counts())

print("\nTotal:", len(major_df))

major_status
valid          90846
invalid         1885
recoverable       11
Name: count, dtype: int64

Total: 92742


In [8]:
# Look for suspicious responses that are still marked "valid"

currently_valid = major_df[
    major_df["major_status"] == "valid"
]

# Responses containing no letters at all
no_letters = currently_valid[
    ~currently_valid["major_clean"].str.contains(
        r"[A-Za-zÀ-ÿ]",
        regex=True,
        na=False
    )
]["major_clean"].value_counts()

print("VALID RESPONSES WITH NO LETTERS:")
print(no_letters.to_string())

VALID RESPONSES WITH NO LETTERS:
major_clean
&#25968;&#23398;                                 1
&#1575;&#1604;&#1607;&#1606;&#1583;&#1587;&#1    1
&#36896;&#22290;                                 1
&#1578;&#1580;&#1575;&#1585;&#1577; &#1575;&#    1
--------                                         1
&#49900;&#47532;&#54617;, &#50672;&#44537;&#5    1
&#1056;&#1072;&#1079;&#1088;&#1072;&#1073;&#1    1
&#44221;&#50689;                                 1
&#25991;&#23398;                                 1
&#27231;&#22120;&#24037;&#31243;                 1
&#24037;&#19994;&#35774;&#35745;                 1
&#46356;&#51088;&#51064;                         1
&#20250;&#35745;                                 1
&#36786;&#23398;                                 1
&#24515;&#29702;&#23398;&#65292;&#33521;&#358    1
&#19981;&#30693;&#36947;                         1
&#1589;&#1610;&#1583;&#1604;&#1577;              1
&#1602;&#1575;&#1606;&#1608;&#1606;              1
&#21307;&#23398;                     

In [9]:
possible_junk = {
    "yes",
    "nice",
    "good",
    "lol",
    "poop",
    "what",
    "all",
    "both",
    "full",
    "open"
}

print("\nPOSSIBLE JUNK STILL MARKED VALID:")

print(
    currently_valid.loc[
        currently_valid["major_clean"].isin(possible_junk),
        "major_clean"
    ].value_counts()
)


POSSIBLE JUNK STILL MARKED VALID:
major_clean
yes     35
nice     2
all      2
poop     1
what     1
full     1
good     1
open     1
both     1
lol      1
Name: count, dtype: Int64


In [10]:
print("\nVERY SHORT VALID RESPONSES:")

print(
    currently_valid.loc[
        currently_valid["major_clean"].str.len() <= 2,
        "major_clean"
    ]
    .value_counts()
    .to_string()
)


VERY SHORT VALID RESPONSES:
major_clean
it    234
hr     71
cs     15
aa     13
ba     11
rn      9
pr      8
ir      7
ma      4
cj      4
x       4
ot      4
ee      4
hm      3
ca      2
rt      2
ag      2
pe      2
p       2
a       2
sw      2
l       2
tv      2
bm      2
e       2
kl      1
fu      1
m       1
b       1
xx      1
me      1
ge      1
re      1
ux      1
pt      1
dd      1
c       1
on      1
gs      1
si      1
d       1
io      1
ec      1
rh      1
ai      1
gb      1
md      1
ed      1
n       1
hk      1
bs      1
et      1
uh      1
ce      1


Unicode-safe check

In [11]:
# Check responses that contain no letters in any language

has_letter = major_df["major_clean"].str.contains(
    r"[^\W\d_]",
    regex=True,
    na=False
)

no_real_letters = major_df[
    (major_df["major_status"] == "valid") &
    (~has_letter)
]["major_clean"].value_counts()

print(no_real_letters.to_string())

major_clean
&#25968;&#23398;                                 1
&#1575;&#1604;&#1607;&#1606;&#1583;&#1587;&#1    1
&#36896;&#22290;                                 1
&#1578;&#1580;&#1575;&#1585;&#1577; &#1575;&#    1
--------                                         1
&#49900;&#47532;&#54617;, &#50672;&#44537;&#5    1
&#1056;&#1072;&#1079;&#1088;&#1072;&#1073;&#1    1
&#44221;&#50689;                                 1
&#25991;&#23398;                                 1
&#27231;&#22120;&#24037;&#31243;                 1
&#24037;&#19994;&#35774;&#35745;                 1
&#46356;&#51088;&#51064;                         1
&#20250;&#35745;                                 1
&#36786;&#23398;                                 1
&#24515;&#29702;&#23398;&#65292;&#33521;&#358    1
&#19981;&#30693;&#36947;                         1
&#1589;&#1610;&#1583;&#1604;&#1577;              1
&#1602;&#1575;&#1606;&#1608;&#1606;              1
&#21307;&#23398;                                 1
-------------------

Mark punctuation-only and numeric-only answers invalid

In [12]:
punctuation_only = major_df["major_clean"].str.fullmatch(
    r"[\W_]+",
    na=False
)

numeric_only = major_df["major_clean"].str.fullmatch(
    r"\d+(?:[.,]\d+)?",
    na=False
)

major_df.loc[
    (major_df["major_status"] == "valid") &
    (punctuation_only | numeric_only),
    "major_status"
] = "invalid"

Remove the Chinese “I don't know”

In [13]:
major_df.loc[
    major_df["major_clean"] == "不知道",
    "major_status"
] = "invalid"

Remove school-only answers we missed

In [14]:
school_only = {
    "high school",
    "senior high school",
    "in high school"
}

major_df.loc[
    major_df["major_clean"].isin(school_only),
    "major_status"
] = "invalid"

Remove obvious word junk

In [15]:
junk_values = {
    "yes",
    "nice",
    "good",
    "lol",
    "poop",
    "what",
    "all",
    "both",
    "full",
    "open",

    # meaningless one/two-character answers
    "x",
    "xx",
    "p",
    "a",
    "l",
    "e",
    "m",
    "b",
    "c",
    "d",
    "n"
}

major_df.loc[
    major_df["major_clean"].isin(junk_values),
    "major_status"
] = "invalid"

Standardize only clear short abbreviations

In [16]:
short_aliases = {
    "it": "information technology",
    "hr": "human resources",
    "cs": "computer science",
    "rn": "nursing",
    "pr": "public relations",
    "ir": "international relations",
    "cj": "criminal justice",
    "ot": "occupational therapy",
    "ee": "electrical engineering",
    "sw": "social work",
    "pt": "physical therapy",
    "md": "medicine"
}

for original, replacement in short_aliases.items():

    mask = (
        (major_df["major_status"] == "valid") &
        (major_df["major_clean"] == original)
    )

    major_df.loc[
        mask,
        "major_status"
    ] = "recoverable"

    major_df.loc[
        mask,
        "major_standardized"
    ] = replacement

Check the status counts

In [17]:
print(major_df["major_status"].value_counts())
print("\nTotal:", len(major_df))

major_status
valid          90394
invalid         1977
recoverable      371
Name: count, dtype: int64

Total: 92742


Check that obvious junk is gone

In [18]:
possible_junk = {
    "yes", "nice", "good", "lol",
    "poop", "what", "all", "both",
    "full", "open"
}

remaining_junk = major_df.loc[
    (major_df["major_status"] == "valid") &
    (major_df["major_clean"].isin(possible_junk)),
    "major_clean"
].value_counts()

print(remaining_junk)

Series([], Name: count, dtype: Int64)


Create the usable major dataframe

In [19]:
major_valid_df = major_df[
    major_df["major_status"] != "invalid"
].copy()

print("Original:", len(major_df))
print("Usable major rows:", len(major_valid_df))

Original: 92742
Usable major rows: 90765


Verify the usable dataset

In [20]:
print("\nSTATUS:")
print(major_valid_df["major_status"].value_counts())

print(
    "\nMissing standardized majors:",
    major_valid_df["major_standardized"].isna().sum()
)

print(
    "Unique standardized majors:",
    major_valid_df["major_standardized"].nunique()
)


STATUS:
major_status
valid          90394
recoverable      371
Name: count, dtype: int64

Missing standardized majors: 0
Unique standardized majors: 11627


In [21]:
print(major_df["major_status"].value_counts())
print("Total:", len(major_df))
print("Usable major rows:", len(major_valid_df))

major_status
valid          90394
invalid         1977
recoverable      371
Name: count, dtype: int64
Total: 92742
Usable major rows: 90765


In [22]:
missing_std = major_valid_df[
    major_valid_df["major_standardized"].isna()
][
    ["major", "major_clean", "major_status"]
]

print(missing_std.to_string())

Empty DataFrame
Columns: [major, major_clean, major_status]
Index: []


In [23]:
major_valid_df["major_standardized"] = (
    major_valid_df["major_standardized"]
    .fillna(major_valid_df["major_clean"])
)

In [24]:
print(
    "Missing standardized majors:",
    major_valid_df["major_standardized"].isna().sum()
)

Missing standardized majors: 0


In [25]:
mask_na_major = (
    major_df["major_clean"].isna() &
    major_df["major"].astype(str).str.strip().str.lower().isin(
        ["n/a", "n.a", "n.a.", "na"]
    )
)

print("Rows found:", mask_na_major.sum())

major_df.loc[
    mask_na_major,
    "major_status"
] = "invalid"

Rows found: 0


In [26]:
major_valid_df = major_df[
    major_df["major_status"] != "invalid"
].copy()

In [27]:
print("Usable rows:", len(major_valid_df))

print(
    "Missing major_clean:",
    major_valid_df["major_clean"].isna().sum()
)

print(
    "Missing major_standardized:",
    major_valid_df["major_standardized"].isna().sum()
)

Usable rows: 90765
Missing major_clean: 0
Missing major_standardized: 0


In [28]:
print(major_df["major_status"].value_counts())
print("Total:", len(major_df))

major_status
valid          90394
invalid         1977
recoverable      371
Name: count, dtype: int64
Total: 92742


In [29]:
major_aliases = {
    # Psychology
    "psych": "psychology",
    "psyc": "psychology",
    "psy": "psychology",
    "pych": "psychology",
    "psycology": "psychology",
    "pyschology": "psychology",
    "psycholgy": "psychology",
    "pschology": "psychology",

    # Computer Science
    "cs": "computer science",
    "comp sci": "computer science",
    "computer sciences": "computer science",

    # Information Technology
    "it": "information technology",
    "i.t": "information technology",
    "i.t.": "information technology",
    "i t": "information technology",

    # Biology
    "bio": "biology",
    "biological sciences": "biology",

    # Economics
    "econ": "economics",
    "eco": "economics",
    "economic": "economics",
    "economy": "economics",

    # Mathematics
    "math": "mathematics",
    "maths": "mathematics",

    # Chemistry
    "chem": "chemistry",

    # Business
    "buisness": "business",

    # Business Administration
    "business admin": "business administration",
    "bba": "business administration",
    "bsba": "business administration",

    # Accounting
    "accountancy": "accounting",
    "acct": "accounting",
    "acc": "accounting",
    "accy": "accounting",
    "account": "accounting",

    # Human Resources
    "hr": "human resources",
    "human resource": "human resources",
    "hrm": "human resource management",
    "human resources management": "human resource management",

    # Nursing
    "rn": "nursing",
    "r n": "nursing",
    "bsn": "nursing",
    "lpn": "nursing",
    "nurse": "nursing",

    # Medicine
    "md": "medicine",
    "m.d.": "medicine",
    "mbbs": "medicine",

    # Veterinary
    "vet": "veterinary medicine",

    # Marketing
    "mkt": "marketing",

    # Management
    "mgmt": "management",

    # Education
    "edu": "education",

    # Architecture
    "arch": "architecture",

    # Mechanical Engineering
    "mech": "mechanical engineering",

    # Electrical Engineering
    "ee": "electrical engineering",

    # Criminal Justice
    "cj": "criminal justice",
    "crj": "criminal justice",

    # Social Work
    "sw": "social work",

    # Public Relations
    "pr": "public relations",

    # Occupational Therapy
    "ot": "occupational therapy",

    # Physical Therapy
    "pt": "physical therapy",
    "physiotherapy": "physical therapy",

    # Speech-Language Pathology
    "slp": "speech-language pathology",

    # GIS
    "gis": "geographic information systems",

    # Communication
    "communications": "communication",
    "communication studies": "communication",
    "mass communication": "communication",
    "mass communications": "communication",

    # Counseling
    "counselling": "counseling",

    # Health Sciences
    "health science": "health sciences",

    # Fine Arts
    "fine art": "fine arts",

    # Social Sciences
    "social science": "social sciences",

    # Electronics Engineering
    "electronic engineering": "electronics engineering",

    # Cybersecurity
    "cybersecurity": "cyber security",

    # Sports Management
    "sport management": "sports management",

    # Pre-med
    "pre med": "pre-med",

    # English teaching
    "esl": "english as a second language",
    "tefl": "teaching english as a foreign language",
    "tesl": "teaching english as a second language"
}

In [30]:
before_standardization = major_valid_df["major_standardized"].copy()

major_valid_df["major_standardized"] = (
    major_valid_df["major_standardized"]
    .replace(major_aliases)
)

changed = (
    before_standardization !=
    major_valid_df["major_standardized"]
)

print("Rows standardized:", changed.sum())
print("Unique BEFORE:", before_standardization.nunique())
print("Unique AFTER:", major_valid_df["major_standardized"].nunique())

Rows standardized: 3061
Unique BEFORE: 11627
Unique AFTER: 11565


In [31]:
changes = (
    major_valid_df.loc[
        changed,
        ["major_clean", "major_standardized"]
    ]
    .drop_duplicates()
    .sort_values(["major_standardized", "major_clean"])
)

print(changes.to_string(index=False))

               major_clean                     major_standardized
                       acc                             accounting
                   account                             accounting
               accountancy                             accounting
                      acct                             accounting
                      accy                             accounting
                      arch                           architecture
                       bio                                biology
       biological sciences                                biology
                  buisness                               business
                       bba                business administration
                      bsba                business administration
            business admin                business administration
                      chem                              chemistry
     communication studies                          communication
          

In [32]:
print(
    major_valid_df["major_standardized"]
    .value_counts()
    .head(100)
    .to_string()
)

major_standardized
psychology                   14141
business                      3924
english                       3221
nursing                       2444
biology                       2168
education                     1947
accounting                    1525
engineering                   1497
computer science              1279
economics                     1263
communication                 1252
law                           1111
civil engineering             1104
history                        976
criminal justice               933
business administration        927
marketing                      922
sociology                      919
management                     848
mathematics                    836
medicine                       804
finance                        786
mechanical engineering         769
counseling                     731
social work                    698
business management            694
political science              684
science                        592
c

## Major category mapping

Standardized free-text majors are mapped to **15 broad academic categories** using a series of high-confidence mapping passes.

The passes are kept separate in this notebook to make the iterative cleaning process transparent. Later passes add mappings for additional spelling variants and rare labels without changing mappings that were already considered reliable.

Uncertain labels remain `unmapped`. The goal is not 100% coverage; the goal is a cleaner supervised target with less label noise.


In [33]:
major_valid_df["major_category"] = "unmapped"

In [34]:
category_map = {

    # ========================================================
    # SOCIAL & BEHAVIORAL SCIENCES
    # ========================================================
    "psychology": "social & behavioral sciences",
    "sociology": "social & behavioral sciences",
    "political science": "social & behavioral sciences",
    "anthropology": "social & behavioral sciences",
    "social sciences": "social & behavioral sciences",
    "politics": "social & behavioral sciences",

    # ========================================================
    # BUSINESS, ECONOMICS & MANAGEMENT
    # ========================================================
    "business": "business, economics & management",
    "business administration": "business, economics & management",
    "business management": "business, economics & management",
    "management": "business, economics & management",
    "marketing": "business, economics & management",
    "accounting": "business, economics & management",
    "economics": "business, economics & management",
    "finance": "business, economics & management",
    "commerce": "business, economics & management",
    "human resources": "business, economics & management",
    "human resource management": "business, economics & management",
    "international business": "business, economics & management",
    "administration": "business, economics & management",
    "mba": "business, economics & management",

    # ========================================================
    # LANGUAGES & HUMANITIES
    # ========================================================
    "english": "languages & humanities",
    "history": "languages & humanities",
    "philosophy": "languages & humanities",
    "liberal arts": "languages & humanities",
    "liberal studies": "languages & humanities",
    "linguistics": "languages & humanities",
    "literature": "languages & humanities",
    "english literature": "languages & humanities",
    "spanish": "languages & humanities",
    "french": "languages & humanities",
    "theology": "languages & humanities",
    "humanities": "languages & humanities",
    "art history": "languages & humanities",

    # ========================================================
    # HEALTH & WELFARE
    # ========================================================
    "nursing": "health & welfare",
    "medicine": "health & welfare",
    "medical": "health & welfare",
    "health": "health & welfare",
    "health sciences": "health & welfare",
    "pharmacy": "health & welfare",
    "public health": "health & welfare",
    "dentistry": "health & welfare",
    "physical therapy": "health & welfare",
    "occupational therapy": "health & welfare",
    "speech-language pathology": "health & welfare",
    "nutrition": "health & welfare",
    "kinesiology": "health & welfare",
    "exercise science": "health & welfare",
    "counseling": "health & welfare",
    "social work": "health & welfare",
    "human services": "health & welfare",
    "pre-med": "health & welfare",

    # ========================================================
    # NATURAL SCIENCES & MATHEMATICS
    # ========================================================
    "biology": "natural sciences & mathematics",
    "chemistry": "natural sciences & mathematics",
    "physics": "natural sciences & mathematics",
    "mathematics": "natural sciences & mathematics",
    "biochemistry": "natural sciences & mathematics",
    "microbiology": "natural sciences & mathematics",
    "neuroscience": "natural sciences & mathematics",
    "biotechnology": "natural sciences & mathematics",
    "geology": "natural sciences & mathematics",
    "geography": "natural sciences & mathematics",
    "environmental science": "natural sciences & mathematics",

    # ========================================================
    # EDUCATION
    # ========================================================
    "education": "education",
    "teaching": "education",
    "early childhood education": "education",
    "elementary education": "education",

    # ========================================================
    # ENGINEERING & BUILT ENVIRONMENT
    # ========================================================
    "engineering": "engineering & built environment",
    "civil engineering": "engineering & built environment",
    "mechanical engineering": "engineering & built environment",
    "electrical engineering": "engineering & built environment",
    "chemical engineering": "engineering & built environment",
    "computer engineering": "engineering & built environment",
    "industrial engineering": "engineering & built environment",
    "biomedical engineering": "engineering & built environment",
    "electronics engineering": "engineering & built environment",
    "architecture": "engineering & built environment",

    # ========================================================
    # COMPUTING & IT
    # ========================================================
    "computer science": "computing & it",
    "information technology": "computing & it",
    "software engineering": "computing & it",
    "cyber security": "computing & it",
    "information systems": "computing & it",

    # ========================================================
    # LAW & CRIMINAL JUSTICE
    # ========================================================
    "law": "law & criminal justice",
    "criminal justice": "law & criminal justice",
    "criminology": "law & criminal justice",

    # ========================================================
    # COMMUNICATION & MEDIA
    # ========================================================
    "communication": "communication & media",
    "journalism": "communication & media",
    "public relations": "communication & media",
    "film": "communication & media",
    "advertising": "communication & media",
    "media": "communication & media",

    # ========================================================
    # ARTS & DESIGN
    # ========================================================
    "art": "arts & design",
    "arts": "arts & design",
    "music": "arts & design",
    "design": "arts & design",
    "graphic design": "arts & design",
    "fine arts": "arts & design",
    "photography": "arts & design",
    "theatre": "arts & design",
    "interior design": "arts & design",

    # ========================================================
    # HOSPITALITY & SERVICES
    # ========================================================
    "hospitality": "hospitality & services",
    "tourism": "hospitality & services",

    # ========================================================
    # AGRICULTURE & ANIMAL SCIENCES
    # ========================================================
    "agriculture": "agriculture & animal sciences",
    "animal science": "agriculture & animal sciences",
    "veterinary medicine": "agriculture & animal sciences",

    # ========================================================
    # GENERAL / INTERDISCIPLINARY
    # ========================================================
    "science": "general / interdisciplinary",
    "general studies": "general / interdisciplinary"
}

In [35]:
mapped_categories = (
    major_valid_df["major_standardized"]
    .map(category_map)
)

major_valid_df.loc[
    mapped_categories.notna(),
    "major_category"
] = mapped_categories[mapped_categories.notna()]

In [36]:
total = len(major_valid_df)

mapped = (
    major_valid_df["major_category"] != "unmapped"
).sum()

unmapped = (
    major_valid_df["major_category"] == "unmapped"
).sum()

print("Total respondents:", total)
print("Mapped respondents:", mapped)
print("Unmapped respondents:", unmapped)
print(f"Coverage: {mapped / total * 100:.2f}%")

Total respondents: 90765
Mapped respondents: 64696
Unmapped respondents: 26069
Coverage: 71.28%


In [37]:
print(
    major_valid_df["major_category"]
    .value_counts()
)

major_category
unmapped                            26069
social & behavioral sciences        16477
business, economics & management    12122
health & welfare                     7018
languages & humanities               6002
engineering & built environment      5491
natural sciences & mathematics       5192
education                            2440
arts & design                        2311
law & criminal justice               2234
computing & it                       2092
communication & media                2012
general / interdisciplinary           835
hospitality & services                266
agriculture & animal sciences         204
Name: count, dtype: int64


In [38]:
unmapped_counts = (
    major_valid_df.loc[
        major_valid_df["major_category"] == "unmapped",
        "major_standardized"
    ]
    .value_counts()
)

print(
    unmapped_counts
    .head(100)
    .to_string()
)

major_standardized
international relations              239
human development                     85
sports management                     84
biomedical science                    84
special education                     83
hospitality management                82
languages                             82
physical education                    80
culinary arts                         80
international studies                 76
child development                     76
aerospace engineering                 75
music education                       74
culinary                              73
zoology                               73
statistics                            70
rehabilitation services               67
doctor                                67
general                               66
public administration                 65
computer                              62
fashion merchandising                 60
computing                             60
archaeology                           

Second mapping pass

In [39]:
category_map_2 = {

    # ========================================================
    # PUBLIC & INTERNATIONAL AFFAIRS
    # ========================================================
    "international relations": "public & international affairs",
    "international studies": "public & international affairs",
    "public administration": "public & international affairs",

    # ========================================================
    # SOCIAL & BEHAVIORAL SCIENCES
    # ========================================================
    "human development": "social & behavioral sciences",
    "child development": "social & behavioral sciences",
    "behavioral science": "social & behavioral sciences",
    "clinical psychology": "social & behavioral sciences",
    "counseling psychology": "social & behavioral sciences",
    "forensic psychology": "social & behavioral sciences",

    # ========================================================
    # BUSINESS, ECONOMICS & MANAGEMENT
    # ========================================================
    "sports management": "business, economics & management",
    "fashion merchandising": "business, economics & management",
    "accountant": "business, economics & management",
    "accounting and finance": "business, economics & management",
    "entrepreneurship": "business, economics & management",
    "business studies": "business, economics & management",

    # ========================================================
    # HEALTH & WELFARE
    # ========================================================
    "biomedical science": "health & welfare",
    "biomedical sciences": "health & welfare",
    "rehabilitation services": "health & welfare",
    "physiology": "health & welfare",
    "medical technology": "health & welfare",
    "athletic training": "health & welfare",
    "radiology": "health & welfare",
    "school counseling": "health & welfare",
    "speech pathology": "health & welfare",
    "mental health counseling": "health & welfare",
    "clinical mental health counseling": "health & welfare",
    "dietetics": "health & welfare",
    "medical assistant": "health & welfare",
    "medical assisting": "health & welfare",
    "dental hygiene": "health & welfare",
    "healthcare": "health & welfare",
    "communication disorders": "health & welfare",

    # Respondents answered with occupations rather than majors,
    # but the broad academic field is still clear.
    "doctor": "health & welfare",
    "medic": "health & welfare",

    # ========================================================
    # EDUCATION
    # ========================================================
    "special education": "education",
    "physical education": "education",
    "music education": "education",
    "teacher": "education",
    "general education": "education",
    "english education": "education",
    "early childhood": "education",

    # ========================================================
    # NATURAL SCIENCES & MATHEMATICS
    # ========================================================
    "zoology": "natural sciences & mathematics",
    "statistics": "natural sciences & mathematics",
    "environmental studies": "natural sciences & mathematics",
    "marine biology": "natural sciences & mathematics",
    "food science": "natural sciences & mathematics",
    "astronomy": "natural sciences & mathematics",
    "forensic science": "natural sciences & mathematics",
    "forensics": "natural sciences & mathematics",

    # ========================================================
    # ENGINEERING & BUILT ENVIRONMENT
    # ========================================================
    "aerospace engineering": "engineering & built environment",
    "aeronautical engineering": "engineering & built environment",
    "environmental engineering": "engineering & built environment",
    "electronics": "engineering & built environment",
    "construction management": "engineering & built environment",
    "engineer": "engineering & built environment",
    "architect": "engineering & built environment",
    "urban planning": "engineering & built environment",
    "quantity surveying": "engineering & built environment",

    # ========================================================
    # COMPUTING & IT
    # ========================================================
    "computer": "computing & it",
    "computers": "computing & it",
    "computing": "computing & it",
    "computer information systems": "computing & it",

    # ========================================================
    # LANGUAGES & HUMANITIES
    # ========================================================
    "languages": "languages & humanities",
    "language": "languages & humanities",
    "archaeology": "languages & humanities",
    "japanese": "languages & humanities",
    "translation": "languages & humanities",
    "classics": "languages & humanities",
    "religion": "languages & humanities",
    "german": "languages & humanities",
    "chinese": "languages & humanities",
    "religious studies": "languages & humanities",
    "creative writing": "languages & humanities",
    "library science": "languages & humanities",

    # ========================================================
    # ARTS & DESIGN
    # ========================================================
    "animation": "arts & design",
    "fashion design": "arts & design",
    "dance": "arts & design",
    "theater": "arts & design",
    "fashion": "arts & design",
    "drama": "arts & design",
    "studio art": "arts & design",
    "industrial design": "arts & design",

    # ========================================================
    # COMMUNICATION & MEDIA
    # ========================================================
    "media studies": "communication & media",

    # ========================================================
    # LAW & CRIMINAL JUSTICE
    # ========================================================
    "paralegal": "law & criminal justice",
    "legal studies": "law & criminal justice",

    # ========================================================
    # HOSPITALITY & SERVICES
    # ========================================================
    "hospitality management": "hospitality & services",
    "culinary arts": "hospitality & services",
    "culinary": "hospitality & services",
    "hotel management": "hospitality & services",
    "cosmetology": "hospitality & services",

    # ========================================================
    # GENERAL / INTERDISCIPLINARY
    # ========================================================
    "general": "general / interdisciplinary",
    "applied science": "general / interdisciplinary",
    "interdisciplinary studies": "general / interdisciplinary"
}

In [40]:
new_categories = (
    major_valid_df["major_standardized"]
    .map(category_map_2)
)

mask = (
    major_valid_df["major_category"].eq("unmapped")
    & new_categories.notna()
)

major_valid_df.loc[
    mask,
    "major_category"
] = new_categories[mask]

print("New respondents mapped:", mask.sum())

New respondents mapped: 4894


In [41]:
total = len(major_valid_df)

mapped = (
    major_valid_df["major_category"] != "unmapped"
).sum()

unmapped = (
    major_valid_df["major_category"] == "unmapped"
).sum()

print("Total respondents:", total)
print("Mapped respondents:", mapped)
print("Unmapped respondents:", unmapped)
print(f"Coverage: {mapped / total * 100:.2f}%")

Total respondents: 90765
Mapped respondents: 69590
Unmapped respondents: 21175
Coverage: 76.67%


In [42]:
print(
    major_valid_df["major_category"]
    .value_counts()
)

major_category
unmapped                            21175
social & behavioral sciences        16808
business, economics & management    12427
health & welfare                     7956
languages & humanities               6577
engineering & built environment      5903
natural sciences & mathematics       5575
education                            2836
arts & design                        2689
law & criminal justice               2325
computing & it                       2291
communication & media                2044
general / interdisciplinary          1003
hospitality & services                572
public & international affairs        380
agriculture & animal sciences         204
Name: count, dtype: int64


In [43]:
unmapped_counts = (
    major_valid_df.loc[
        major_valid_df["major_category"] == "unmapped",
        "major_standardized"
    ]
    .value_counts()
)

print(
    unmapped_counts
    .head(100)
    .to_string()
)

major_standardized
aviation                                     55
actuarial science                            52
bussiness                                    46
technology                                   40
psicologia                                   40
sports                                       38
cpa                                          32
horticulture                                 31
healthcare administration                    31
leadership                                   31
american studies                             31
family studies                               31
electronics and communication engineering    31
human resource development                   30
civil                                        29
writing                                      29
acting                                       29
computer programming                         28
business and management                      28
illustration                                 28
performing arts      

In [44]:
mask_not_yet = (
    major_valid_df["major_standardized"] == "not yet"
)

print("Rows removed:", mask_not_yet.sum())

major_valid_df.loc[
    mask_not_yet,
    "major_status"
] = "invalid"

major_valid_df = major_valid_df[
    major_valid_df["major_status"] != "invalid"
].copy()

print("Rows remaining:", len(major_valid_df))

Rows removed: 21
Rows remaining: 90744


In [45]:
major_aliases_3 = {
    # Psychology
    "phychology": "psychology",
    "psicologia": "psychology",

    # Business
    "bussiness": "business",
    "business adminstration": "business administration",
    "business managment": "business management",

    # Accounting
    "accounts": "accounting",

    # Mathematics
    "mathematic": "mathematics",

    # Biology
    "biological science": "biology",
    "life science": "life sciences",

    # Healthcare
    "health care": "healthcare",
    "medical assisting": "medical assistant",

    # Information Systems
    "information system": "information systems",

    # Veterinary
    "veterinary": "veterinary medicine"
}

In [46]:
before_3 = major_valid_df["major_standardized"].copy()

major_valid_df["major_standardized"] = (
    major_valid_df["major_standardized"]
    .replace(major_aliases_3)
)

changed_3 = (
    before_3 != major_valid_df["major_standardized"]
)

print("Rows standardized:", changed_3.sum())
print("Unique BEFORE:", before_3.nunique())
print(
    "Unique AFTER:",
    major_valid_df["major_standardized"].nunique()
)

Rows standardized: 362
Unique BEFORE: 11564
Unique AFTER: 11551


In [47]:
category_map_3 = {

    # ========================================================
    # BUSINESS, ECONOMICS & MANAGEMENT
    # ========================================================
    "actuarial science":
        "business, economics & management",

    "business and management":
        "business, economics & management",

    "business marketing":
        "business, economics & management",

    "leadership":
        "business, economics & management",

    "human resource development":
        "business, economics & management",

    "logistics":
        "business, economics & management",

    "supply chain management":
        "business, economics & management",

    "organizational leadership":
        "business, economics & management",

    "management studies":
        "business, economics & management",

    "financial management":
        "business, economics & management",

    "business economics":
        "business, economics & management",

    # ========================================================
    # SOCIAL & BEHAVIORAL SCIENCES
    # ========================================================
    "family studies":
        "social & behavioral sciences",

    "industrial psychology":
        "social & behavioral sciences",

    "psychology and sociology":
        "social & behavioral sciences",

    "human development and family studies":
        "social & behavioral sciences",

    "educational psychology":
        "social & behavioral sciences",

    # ========================================================
    # HEALTH & WELFARE
    # ========================================================
    "healthcare administration":
        "health & welfare",

    "sports medicine":
        "health & welfare",

    "sport science":
        "health & welfare",

    "rehabilitation counseling":
        "health & welfare",

    "social services":
        "health & welfare",

    "medical science":
        "health & welfare",

    "pharmacology":
        "health & welfare",

    "biomedical":
        "health & welfare",

    "healthcare management":
        "health & welfare",

    "medical field":
        "health & welfare",

    "pre-nursing":
        "health & welfare",

    "health services management":
        "health & welfare",

    "dental":
        "health & welfare",

    "communication sciences and disorders":
        "health & welfare",

    # ========================================================
    # NATURAL SCIENCES & MATHEMATICS
    # ========================================================
    "life sciences":
        "natural sciences & mathematics",

    "molecular biology":
        "natural sciences & mathematics",

    "ecology":
        "natural sciences & mathematics",

    "genetics":
        "natural sciences & mathematics",

    "biological science":
        "natural sciences & mathematics",

    "astrophysics":
        "natural sciences & mathematics",

    # ========================================================
    # ENGINEERING & BUILT ENVIRONMENT
    # ========================================================
    "electronics and communication engineering":
        "engineering & built environment",

    "construction":
        "engineering & built environment",

    "bioengineering":
        "engineering & built environment",

    "marine engineering":
        "engineering & built environment",

    "petroleum engineering":
        "engineering & built environment",

    "mechatronics engineering":
        "engineering & built environment",

    # ========================================================
    # COMPUTING & IT
    # ========================================================
    "computer programming":
        "computing & it",

    "programming":
        "computing & it",

    "management information systems":
        "computing & it",

    "information management":
        "computing & it",

    "computer science engineering":
        "computing & it",

    "informatics":
        "computing & it",

    "information system":
        "computing & it",

    "information systems":
        "computing & it",

    # ========================================================
    # EDUCATION
    # ========================================================
    "educational studies":
        "education",

    "art education":
        "education",

    "pedagogy":
        "education",

    "values education":
        "education",

    # ========================================================
    # LANGUAGES & HUMANITIES
    # ========================================================
    "american studies":
        "languages & humanities",

    "writing":
        "languages & humanities",

    "filipino":
        "languages & humanities",

    "foreign languages":
        "languages & humanities",

    "modern languages":
        "languages & humanities",

    # ========================================================
    # ARTS & DESIGN
    # ========================================================
    "acting":
        "arts & design",

    "illustration":
        "arts & design",

    "performing arts":
        "arts & design",

    "game design":
        "arts & design",

    "visual arts":
        "arts & design",

    "art and design":
        "arts & design",

    "product design":
        "arts & design",

    # ========================================================
    # COMMUNICATION & MEDIA
    # ========================================================
    "multimedia":
        "communication & media",

    "digital media":
        "communication & media",

    "communication arts":
        "communication & media",

    "film production":
        "communication & media",

    # ========================================================
    # LAW & CRIMINAL JUSTICE
    # ========================================================
    "law enforcement":
        "law & criminal justice",

    "paralegal studies":
        "law & criminal justice",

    # ========================================================
    # PUBLIC & INTERNATIONAL AFFAIRS
    # ========================================================
    "international affairs":
        "public & international affairs",

    "public policy":
        "public & international affairs",

    # ========================================================
    # AGRICULTURE & ANIMAL SCIENCES
    # ========================================================
    "horticulture":
        "agriculture & animal sciences",

    "forestry":
        "agriculture & animal sciences",

    # ========================================================
    # HOSPITALITY & SERVICES
    # ========================================================
    "tourism management":
        "hospitality & services",

    # ========================================================
    # GENERAL / INTERDISCIPLINARY
    # ========================================================
    "sciences":
        "general / interdisciplinary",

    "stem":
        "general / interdisciplinary"
}

In [48]:
new_categories_3 = (
    major_valid_df["major_standardized"]
    .map(category_map_3)
)

mask_3 = (
    major_valid_df["major_category"].eq("unmapped")
    & new_categories_3.notna()
)

major_valid_df.loc[
    mask_3,
    "major_category"
] = new_categories_3[mask_3]

print("New respondents mapped:", mask_3.sum())

New respondents mapped: 1875


In [49]:
total = len(major_valid_df)

mapped = (
    major_valid_df["major_category"] != "unmapped"
).sum()

unmapped = (
    major_valid_df["major_category"] == "unmapped"
).sum()

print("Total respondents:", total)
print("Mapped respondents:", mapped)
print("Unmapped respondents:", unmapped)
print(f"Coverage: {mapped / total * 100:.2f}%")

Total respondents: 90744
Mapped respondents: 71465
Unmapped respondents: 19279
Coverage: 78.75%


In [50]:
print(
    major_valid_df["major_category"]
    .value_counts()
)

major_category
unmapped                            19279
social & behavioral sciences        16932
business, economics & management    12728
health & welfare                     8274
languages & humanities               6702
engineering & built environment      6042
natural sciences & mathematics       5708
education                            2927
arts & design                        2861
computing & it                       2454
law & criminal justice               2370
communication & media                2131
general / interdisciplinary          1053
hospitality & services                597
public & international affairs        429
agriculture & animal sciences         257
Name: count, dtype: int64


In [51]:
unmapped_counts = (
    major_valid_df.loc[
        major_valid_df["major_category"] == "unmapped",
        "major_standardized"
    ]
    .value_counts()
)

print(
    unmapped_counts
    .head(100)
    .to_string()
)

major_standardized
psychology                         67
aviation                           55
business                           46
technology                         40
sports                             38
cpa                                32
civil                              29
food technology                    28
healthcare                         26
veterinary medicine                26
accounting                         25
lawyer                             24
pharmacist                         24
mechanical                         24
cognitive science                  24
business administration            23
business management                23
mathematics                        21
sport                              21
audio engineering                  20
biology                            20
health studies                     19
banking and finance                19
secondary education                19
environmental management           19
electrical                     

In [52]:
# Combine all category mappings we have already created
all_category_maps = {
    **category_map,
    **category_map_2,
    **category_map_3
}

recheck_categories = (
    major_valid_df["major_standardized"]
    .map(all_category_maps)
)

recheck_mask = (
    major_valid_df["major_category"].eq("unmapped")
    & recheck_categories.notna()
)

print("Previously-defined categories recovered:", recheck_mask.sum())

major_valid_df.loc[
    recheck_mask,
    "major_category"
] = recheck_categories[recheck_mask]

Previously-defined categories recovered: 277


In [53]:
new_invalid = {
    "nothing",
    "other"
}

invalid_mask = (
    major_valid_df["major_standardized"]
    .isin(new_invalid)
)

print("New invalid rows:", invalid_mask.sum())

major_valid_df.loc[
    invalid_mask,
    "major_status"
] = "invalid"

major_valid_df = major_valid_df[
    major_valid_df["major_status"] != "invalid"
].copy()

print("Rows remaining:", len(major_valid_df))

New invalid rows: 34
Rows remaining: 90710


In [54]:
major_df.loc[
    major_df["major_clean"].isin(
        ["not yet", "nothing", "other"]
    ),
    "major_status"
] = "invalid"

In [55]:
major_aliases_4 = {

    # Psychology
    "pychology": "psychology",

    # Management
    "managment": "management",
    "hr management": "human resource management",

    # Healthcare
    "health care administration": "healthcare administration",
    "rehab services": "rehabilitation services",
    "pre nursing": "pre-nursing",
    "clinical mental health": "clinical mental health counseling",

    # Social work
    "social worker": "social work",
    "human service": "human services",

    # Political science
    "political sciences": "political science",

    # Art
    "visual art": "visual arts",
    "history of art": "art history",

    # Veterinary
    "veterinarian": "veterinary medicine",

    # Engineering
    "mechanical engineer": "mechanical engineering",
    "computer engineer": "computer engineering",

    # Speech-language pathology
    "speech language pathology": "speech-language pathology",

    # Environmental science
    "environmental sciences": "environmental science"
}

before_4 = major_valid_df["major_standardized"].copy()

major_valid_df["major_standardized"] = (
    major_valid_df["major_standardized"]
    .replace(major_aliases_4)
)

changed_4 = (
    before_4 != major_valid_df["major_standardized"]
)

print("Rows standardized:", changed_4.sum())
print("Unique BEFORE:", before_4.nunique())
print("Unique AFTER:", major_valid_df["major_standardized"].nunique())

Rows standardized: 272
Unique BEFORE: 11549
Unique AFTER: 11532


In [56]:
recheck_categories = (
    major_valid_df["major_standardized"]
    .map(all_category_maps)
)

recheck_mask = (
    major_valid_df["major_category"].eq("unmapped")
    & recheck_categories.notna()
)

print("Recovered after standardization:", recheck_mask.sum())

major_valid_df.loc[
    recheck_mask,
    "major_category"
] = recheck_categories[recheck_mask]

Recovered after standardization: 272


In [57]:
category_map_4 = {

    # ========================================================
    # BUSINESS, ECONOMICS & MANAGEMENT
    # ========================================================
    "banking and finance":
        "business, economics & management",

    "banking":
        "business, economics & management",

    "marketing management":
        "business, economics & management",

    "project management":
        "business, economics & management",

    "real estate":
        "business, economics & management",

    # ========================================================
    # SOCIAL & BEHAVIORAL SCIENCES
    # ========================================================
    "social psychology":
        "social & behavioral sciences",

    "cognitive science":
        "social & behavioral sciences",

    # ========================================================
    # HEALTH & WELFARE
    # ========================================================
    "health studies":
        "health & welfare",

    "art therapy":
        "health & welfare",

    "rehabilitation":
        "health & welfare",

    "community health":
        "health & welfare",

    "mental health":
        "health & welfare",

    "recreational therapy":
        "health & welfare",

    "pharmaceutical sciences":
        "health & welfare",

    "medical laboratory science":
        "health & welfare",

    "optometry":
        "health & welfare",

    "mortuary science":
        "health & welfare",

    # ========================================================
    # EDUCATION
    # ========================================================
    "secondary education":
        "education",

    "adult education":
        "education",

    # ========================================================
    # NATURAL SCIENCES & MATHEMATICS
    # ========================================================
    "botany":
        "natural sciences & mathematics",

    "human biology":
        "natural sciences & mathematics",

    "applied mathematics":
        "natural sciences & mathematics",

    "geophysics":
        "natural sciences & mathematics",

    # ========================================================
    # ENGINEERING & BUILT ENVIRONMENT
    # ========================================================
    "landscape architecture":
        "engineering & built environment",

    "automotive engineering":
        "engineering & built environment",

    "materials engineering":
        "engineering & built environment",

    "mining engineering":
        "engineering & built environment",

    # ========================================================
    # COMPUTING & IT
    # ========================================================
    "computer networking":
        "computing & it",

    "information science":
        "computing & it",

    "computer technology":
        "computing & it",

    # ========================================================
    # LANGUAGES & HUMANITIES
    # ========================================================
    "philology":
        "languages & humanities",

    "russian":
        "languages & humanities",

    # ========================================================
    # ARTS & DESIGN
    # ========================================================
    "theatre arts":
        "arts & design",

    # ========================================================
    # COMMUNICATION & MEDIA
    # ========================================================
    "media arts":
        "communication & media",

    "cinema":
        "communication & media",

    # ========================================================
    # LAW & CRIMINAL JUSTICE
    # ========================================================
    "legal":
        "law & criminal justice",

    # ========================================================
    # PUBLIC & INTERNATIONAL AFFAIRS
    # ========================================================
    "government":
        "public & international affairs",

    "global studies":
        "public & international affairs",

    # ========================================================
    # AGRICULTURE & ANIMAL SCIENCES
    # ========================================================
    "agronomy":
        "agriculture & animal sciences",

    # ========================================================
    # HOSPITALITY & SERVICES
    # ========================================================
    "hotel and restaurant management":
        "hospitality & services",

    "event management":
        "hospitality & services"
}

In [58]:
new_categories_4 = (
    major_valid_df["major_standardized"]
    .map(category_map_4)
)

mask_4 = (
    major_valid_df["major_category"].eq("unmapped")
    & new_categories_4.notna()
)

major_valid_df.loc[
    mask_4,
    "major_category"
] = new_categories_4[mask_4]

print("New respondents mapped:", mask_4.sum())

New respondents mapped: 668


In [59]:
total = len(major_valid_df)

mapped = (
    major_valid_df["major_category"] != "unmapped"
).sum()

unmapped = (
    major_valid_df["major_category"] == "unmapped"
).sum()

print("Total respondents:", total)
print("Mapped respondents:", mapped)
print("Unmapped respondents:", unmapped)
print(f"Coverage: {mapped / total * 100:.2f}%")

Total respondents: 90710
Mapped respondents: 72682
Unmapped respondents: 18028
Coverage: 80.13%


In [60]:
print(
    major_valid_df["major_category"]
    .value_counts()
)

major_category
unmapped                            18028
social & behavioral sciences        17072
business, economics & management    12963
health & welfare                     8576
languages & humanities               6748
engineering & built environment      6145
natural sciences & mathematics       5827
education                            2961
arts & design                        2892
computing & it                       2498
law & criminal justice               2385
communication & media                2161
general / interdisciplinary          1053
hospitality & services                629
public & international affairs        459
agriculture & animal sciences         313
Name: count, dtype: int64


In [61]:
unmapped_counts = (
    major_valid_df.loc[
        major_valid_df["major_category"] == "unmapped",
        "major_standardized"
    ]
    .value_counts()
)

print(
    unmapped_counts
    .head(100)
    .to_string()
)

major_standardized
aviation                                  55
technology                                40
sports                                    38
cpa                                       32
civil                                     29
food technology                           28
pharmacist                                24
mechanical                                24
lawyer                                    24
sport                                     21
audio engineering                         20
psychology and criminal justice           19
electrical                                19
environmental management                  19
social                                    18
welding                                   17
biology and psychology                    17
sports science                            17
social studies                            17
environment                               17
engineering technology                    17
recreation                          

In [62]:
print("Unique unmapped majors:", unmapped_counts.size)

print("\nUnmapped labels appearing only once:")
print((unmapped_counts == 1).sum())

print("\nUnmapped labels appearing fewer than 5 times:")
print((unmapped_counts < 5).sum())

print("\nUnmapped labels appearing at least 10 times:")
print((unmapped_counts >= 10).sum())

print("\nUnmapped labels appearing at least 20 times:")
print((unmapped_counts >= 20).sum())

Unique unmapped majors: 11217

Unmapped labels appearing only once:
8930

Unmapped labels appearing fewer than 5 times:
10635

Unmapped labels appearing at least 10 times:
159

Unmapped labels appearing at least 20 times:
11


In [63]:
all_category_maps = {
    **category_map,
    **category_map_2,
    **category_map_3,
    **category_map_4
}

recheck = (
    major_valid_df["major_standardized"]
    .map(all_category_maps)
)

mask = (
    major_valid_df["major_category"].eq("unmapped")
    & recheck.notna()
)

major_valid_df.loc[
    mask,
    "major_category"
] = recheck[mask]

print("Recovered:", mask.sum())

Recovered: 0


In [64]:
major_aliases_5 = {

    # Psychology
    "psikologi": "psychology",

    # Engineering
    "enginering": "engineering",
    "electrical engineer": "electrical engineering",

    # Computing
    "science computer": "computer science",

    # Languages
    "linguistic": "linguistics",
    "english lit": "english literature",

    # Health
    "pre-medicine": "pre-med",
    "health care management": "healthcare management",

    # Business
    "supply chain": "supply chain management",
    "administração": "administration",

    # Art
    "visual art": "visual arts"
}

before_5 = major_valid_df["major_standardized"].copy()

major_valid_df["major_standardized"] = (
    major_valid_df["major_standardized"]
    .replace(major_aliases_5)
)

changed_5 = (
    before_5 != major_valid_df["major_standardized"]
)

print("Rows standardized:", changed_5.sum())

Rows standardized: 123


In [65]:
recheck = (
    major_valid_df["major_standardized"]
    .map(all_category_maps)
)

mask = (
    major_valid_df["major_category"].eq("unmapped")
    & recheck.notna()
)

major_valid_df.loc[
    mask,
    "major_category"
] = recheck[mask]

print("Recovered after aliases:", mask.sum())

Recovered after aliases: 123


In [66]:
category_map_5 = {

    # SOCIAL & BEHAVIORAL SCIENCES
    "psychology/sociology":
        "social & behavioral sciences",
    "child psychology":
        "social & behavioral sciences",
    "i/o psychology":
        "social & behavioral sciences",
    "biopsychology":
        "social & behavioral sciences",

    # BUSINESS / MANAGEMENT
    "general business":
        "business, economics & management",
    "organizational management":
        "business, economics & management",
    "economics and finance":
        "business, economics & management",
    "office management":
        "business, economics & management",
    "property management":
        "business, economics & management",
    "music business":
        "business, economics & management",
    "aviation management":
        "business, economics & management",

    # HEALTH & WELFARE
    "pharmacist":
        "health & welfare",
    "exercise physiology":
        "health & welfare",
    "health administration":
        "health & welfare",
    "health and wellness":
        "health & welfare",
    "human and social services":
        "health & welfare",
    "allied health":
        "health & welfare",
    "music therapy":
        "health & welfare",
    "respiratory therapy":
        "health & welfare",
    "human services and counseling":
        "health & welfare",
    "medical coding":
        "health & welfare",
    "biomedicine":
        "health & welfare",
    "sports science":
        "health & welfare",

    # NATURAL SCIENCES
    "anatomy":
        "natural sciences & mathematics",

    # ENGINEERING
    "engineering technology":
        "engineering & built environment",
    "manufacturing engineering":
        "engineering & built environment",
    "electrical and electronics engineering":
        "engineering & built environment",
    "nuclear engineering":
        "engineering & built environment",
    "surveying":
        "engineering & built environment",
    "town planning":
        "engineering & built environment",
    "interior architecture":
        "engineering & built environment",

    # COMPUTING & IT
    "ict":
        "computing & it",
    "mis":
        "computing & it",
    "networking":
        "computing & it",
    "software development":
        "computing & it",
    "computer science and engineering":
        "computing & it",
    "it engineering":
        "computing & it",

    # EDUCATION
    "higher education":
        "education",

    # LANGUAGES & HUMANITIES
    "asian studies":
        "languages & humanities",
    "biblical studies":
        "languages & humanities",
    "comparative literature":
        "languages & humanities",
    "cultural studies":
        "languages & humanities",
    "european studies":
        "languages & humanities",

    # ARTS & DESIGN
    "designer":
        "arts & design",
    "communication design":
        "arts & design",

    # COMMUNICATION & MEDIA
    "professional communication":
        "communication & media",
    "visual communication":
        "communication & media",
    "broadcasting":
        "communication & media",

    # LAW
    "lawyer":
        "law & criminal justice",
    "pre law":
        "law & criminal justice",
    "administration of justice":
        "law & criminal justice",

    # PUBLIC / INTERNATIONAL
    "international development":
        "public & international affairs",

    # AGRICULTURE
    "veterinary science":
        "agriculture & animal sciences"
}

In [67]:
new_categories_5 = (
    major_valid_df["major_standardized"]
    .map(category_map_5)
)

mask_5 = (
    major_valid_df["major_category"].eq("unmapped")
    & new_categories_5.notna()
)

major_valid_df.loc[
    mask_5,
    "major_category"
] = new_categories_5[mask_5]

print("New respondents mapped:", mask_5.sum())

New respondents mapped: 679


In [68]:
total = len(major_valid_df)

mapped = (
    major_valid_df["major_category"] != "unmapped"
).sum()

unmapped = (
    major_valid_df["major_category"] == "unmapped"
).sum()

print("Total:", total)
print("Mapped:", mapped)
print("Unmapped:", unmapped)
print(f"Coverage: {mapped / total * 100:.2f}%")

print("\nCATEGORY DISTRIBUTION:")
print(
    major_valid_df["major_category"]
    .value_counts()
)

Total: 90710
Mapped: 73484
Unmapped: 17226
Coverage: 81.01%

CATEGORY DISTRIBUTION:
major_category
unmapped                            17226
social & behavioral sciences        17131
business, economics & management    13076
health & welfare                     8762
languages & humanities               6834
engineering & built environment      6260
natural sciences & mathematics       5838
education                            2973
arts & design                        2918
computing & it                       2583
law & criminal justice               2433
communication & media                2198
general / interdisciplinary          1053
hospitality & services                629
public & international affairs        472
agriculture & animal sciences         324
Name: count, dtype: int64


## Final mapped machine-learning dataset

The complete cleaned major dataset is preserved, including rows whose major could not be mapped confidently.

For supervised machine learning, only rows with a mapped `major_category` are used. This produces the final mapped dataset while retaining the unmapped rows separately for transparency and future improvement.


In [69]:
ml_major_df = major_valid_df[
    major_valid_df["major_category"] != "unmapped"
].copy()

In [70]:
print("ML rows:", len(ml_major_df))
print("Classes:", ml_major_df["major_category"].nunique())


ML rows: 73484
Classes: 15


In [71]:
riasec_features = [
    "R_score",
    "I_score",
    "A_score",
    "S_score",
    "E_score",
    "C_score"
]

print(
    ml_major_df[riasec_features]
    .isna()
    .sum()
)

print("\nScore ranges:")
print(
    ml_major_df[riasec_features]
    .agg(["min", "max", "mean"])
)

R_score    0
I_score    0
A_score    0
S_score    0
E_score    0
C_score    0
dtype: int64

Score ranges:
       R_score   I_score   A_score   S_score  E_score   C_score
min   1.000000  1.000000  1.000000  1.000000  1.00000  1.000000
max   5.000000  5.000000  5.000000  5.000000  5.00000  5.000000
mean  2.095865  3.045012  2.934404  3.400535  2.58887  2.440152


In [72]:
class_counts = (
    ml_major_df["major_category"]
    .value_counts()
)

class_percentages = (
    ml_major_df["major_category"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

class_summary = (
    class_counts
    .to_frame("count")
    .join(
        class_percentages
        .to_frame("percentage")
    )
)

print(class_summary)

                                  count  percentage
major_category                                     
social & behavioral sciences      17131       23.31
business, economics & management  13076       17.79
health & welfare                   8762       11.92
languages & humanities             6834        9.30
engineering & built environment    6260        8.52
natural sciences & mathematics     5838        7.94
education                          2973        4.05
arts & design                      2918        3.97
computing & it                     2583        3.52
law & criminal justice             2433        3.31
communication & media              2198        2.99
general / interdisciplinary        1053        1.43
hospitality & services              629        0.86
public & international affairs      472        0.64
agriculture & animal sciences       324        0.44


## Final preprocessing summary and export

Two datasets are saved:

- `final_dataset.csv`: all usable cleaned major rows, including labels left as `unmapped`;
- `final_dataset_mapped_only.csv`: only rows mapped to one of the 15 academic categories and used for supervised ML.

Download steps are intentionally omitted from the GitHub version so the notebook remains portable and reproducible. In Colab, the files can be downloaded manually from the Files panel after the notebook finishes.


In [73]:
# Final summary
final_total = len(major_valid_df)
final_mapped = len(ml_major_df)
final_unmapped = final_total - final_mapped
mapping_coverage = (final_mapped / final_total * 100) if final_total else 0

print("=" * 60)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 60)
print("Raw respondents:", len(df))
print("RIASEC-clean respondents:", len(riasec_df))
print("Rows with known major:", len(model_df))
print("Usable major rows:", final_total)
print("Mapped ML rows:", final_mapped)
print("Unmapped rows:", final_unmapped)
print("Target categories:", ml_major_df["major_category"].nunique())
print(f"Mapping coverage: {mapping_coverage:.2f}%")

# Final exports
FULL_OUTPUT = "/content/final_dataset.csv"
MAPPED_OUTPUT = "/content/final_dataset_mapped_only.csv"

major_valid_df.to_csv(FULL_OUTPUT, index=False)
ml_major_df.to_csv(MAPPED_OUTPUT, index=False)

print("\nSaved:")
print(FULL_OUTPUT, major_valid_df.shape)
print(MAPPED_OUTPUT, ml_major_df.shape)


FINAL PREPROCESSING SUMMARY
Raw respondents: 145828
RIASEC-clean respondents: 145486
Rows with known major: 92742
Usable major rows: 90710
Mapped ML rows: 73484
Unmapped rows: 17226
Target categories: 15
Mapping coverage: 81.01%

Saved:
/content/final_dataset.csv (90710, 72)
/content/final_dataset_mapped_only.csv (73484, 72)
